# OCAPI Data API — Products (search + product by ID)

Uses the **OCAPI Data API** (via `instance.ocapi`) with **OAuth
client-credentials** to:

- **search products** — `POST /product_search` (OCAPI query DSL), and
- **fetch one product** — `GET /products/{id}`

Responses are parsed into the **generated Pydantic types**
`b2c_tooling_sdk.clients.models.ocapi.ProductSearchResult` and `Product`.

`instance.ocapi` is rooted at `https://{hostname}/s/-/dw/data/v25_6`; calls return
a `ClientResult(data, error, response)` and never raise on 4xx/5xx. Connection
settings come from `../dw.json` (needs `clientId`, `clientSecret`).

In [ ]:
from pathlib import Path

from b2c_tooling_sdk import ResolveConfigOptions, resolve_config
from b2c_tooling_sdk.clients.models.ocapi import Product, ProductSearchResult

DW_JSON = Path("../dw.json").resolve()

config = await resolve_config(options=ResolveConfigOptions(config_path=str(DW_JSON)))
instance = config.create_b2c_instance()
print(f"OCAPI Data API on {instance.config.hostname}")


def local(value):
    """Pick a localized value from an OCAPI localized dict (prefers 'default')."""
    if isinstance(value, dict):
        return value.get("default") or next(iter(value.values()), None)
    return value


def text_of(markup):
    """Render a MarkupText (or plain value) to a short string."""
    if markup is None:
        return None
    return getattr(markup, "markup", None) or getattr(markup, "source", None) or str(markup)


def result_data(result, what):
    """Return parsed data on success; print a friendly message and return None otherwise."""
    if result.error is None and result.data is not None:
        return result.data
    status = result.response.status_code if result.response is not None else "?"
    print(f"{what}: HTTP {status} — {str(result.error)[:200]}")
    return None

In [ ]:
# --- Parameters (edit these) ---
SEARCH_QUERY = "shirt"   # keyword; set to "" (empty) to list all products (match_all_query)
RESULT_LIMIT = 10        # max hits to fetch/print
PRODUCT_ID = None        # None -> use the first search hit; or set a specific product id

## 1. Product search (`POST /product_search`)

The OCAPI query DSL wraps one query type: `text_query` for a keyword search, or
`match_all_query` to page through everything. `select=(**)` returns all hit fields.

In [ ]:
if SEARCH_QUERY:
    query = {"text_query": {"fields": ["id", "name"], "search_phrase": SEARCH_QUERY}}
    label = f'text_query "{SEARCH_QUERY}"'
else:
    query = {"match_all_query": {}}
    label = "match_all_query (all products)"

result = await instance.ocapi.post(
    "/product_search",
    {"body": {"query": query, "select": "(**)", "count": RESULT_LIMIT}},
)

hits = []
data = result_data(result, "product_search")
if data is not None:
    search = ProductSearchResult.model_validate(data)  # <- generated type
    hits = search.hits or []
    print(f"{label}: {search.total} total result(s), showing {len(hits)}:")
    for hit in hits:
        print(f"  • {(hit.id or '?'):<22} {local(hit.name) or '(no name)'}")
    if not hits:
        print("  (no hits — try a different SEARCH_QUERY)")

## 2. Product by ID (`GET /products/{id}`)

Uses `PRODUCT_ID` if set, otherwise the first search hit above. Parsed into the
generated `Product` model (localized `name`/descriptions are dicts keyed by locale).

In [ ]:
product_id = PRODUCT_ID or (hits[0].id if hits else None)
if not product_id:
    print("No product id available — set PRODUCT_ID or broaden SEARCH_QUERY.")
else:
    result = await instance.ocapi.get(
        "/products/{id}",
        {"params": {"path": {"id": product_id}, "query": {"select": "(**)"}}},
    )
    data = result_data(result, f"products/{product_id}")
    if data is not None:
        product = Product.model_validate(data)  # <- generated type
        print("id    :", product.id)
        print("name  :", local(product.name))
        print("brand :", product.brand)
        print("price :", product.price)
        if product.type is not None:
            flags = [k for k, v in product.type.model_dump().items() if v]
            print("type  :", ", ".join(flags) or "(none)")
        desc = text_of(local(product.long_description)) or text_of(local(product.short_description))
        if desc:
            print("desc  :", desc[:160].strip(), "...")